In [35]:
#from data import load_data_v2
#from utils import plot_pool
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from scipy.spatial.distance import cdist
from utils import plot_pool 
import matplotlib.pyplot as plt

import torch
from collections import defaultdict
from torch.utils.data import Subset
from torchvision import datasets
from torchvision.transforms import ToTensor
from sklearn.model_selection import ShuffleSplit


### Load data

In [36]:
def load_data(n_init= 100):

    training_data = datasets.MNIST(
        root="../data",
        train=True,
        download=True,
        transform=ToTensor()
    )
    test_data = datasets.MNIST(
        root="../data",
        train=False,
        download=True,
        transform=ToTensor()
    )


    indices = []
    threshold = 5
    amount = defaultdict(int)
    targets = training_data.targets

    

    for digit in range(10):
        digit_indices = (targets == digit).nonzero(as_tuple=True)[0]
        if digit < threshold:
            selected = digit_indices[:len(digit_indices) // 10]   # keep 10%
            amount["below"] += len(selected)
        else:
            selected = digit_indices[:len(digit_indices) // 100]  # keep 1%
            amount["above"] += len(selected)
        #print(f"For digit {digit} we select {len(selected)}")
        indices.append(selected)

    indices = torch.cat(indices)
    #print(len(indices), amount)

  
    subset_training_data = Subset(training_data, indices=indices)
    #print(len(subset_training_data))

  
    X = training_data.data[indices].float() / 255.0 
    y = training_data.targets[indices]

    X_test = test_data.data.float() / 255.0
    y_test = test_data.targets

  
    seed = 0
    n = len(subset_training_data)
    print(n)

    sss = ShuffleSplit(n_splits=1, train_size=n_init/n, random_state=seed)
    train_idx, pool_idx = next(sss.split(X, y))

   
    data = dict(
        train=dict(
            X=X[train_idx].reshape(len(X[train_idx]), -1),
            y=y[train_idx]
        ),
        pool=dict(
            X=X[pool_idx].reshape(len(X[pool_idx]), -1),
            y=y[pool_idx]
        ),
        test=dict(
            X=X_test.reshape(len(X_test), -1),
            y=y_test
        )
    )
    return data

In [38]:
data = load_data() #n_init
print(f"Train X len: {len(data["train"]["X"])}")
print(f"Train y len: {len(data["train"]["y"])}\n")

print(f"Pool X len: {len(data["pool"]["X"])}")
print(f"Pool y len: {len(data["pool"]["y"])}\n")

print(f"Test X len: {len(data["test"]["X"])}")
print(f"Test y len: {len(data["test"]["y"])}\n")


3350
Train X len: 100
Train y len: 100

Pool X len: 3250
Pool y len: 3250

Test X len: 10000
Test y len: 10000



### Helper functions from solution

In [44]:
#Helper functions from solution
def evaluate_uncertainty(prob, strategy):

    if strategy == 'least confident':
        res = 1 - prob.max(1)
    elif strategy == 'margin':
        ix = np.arange(len(prob))
        p2, p1 = prob.argsort(1)[:, -2:].T
        res = 1 - (prob[ix, p1] - prob[ix, p2])
    elif strategy == 'entropy':
        res = - np.sum(prob * np.log2(prob), axis=1)
    else:
        raise ValueError
    return res

def update_data(data, idx):
    """Update of the data dictionary from `prepare_data` by moving the data
    point with index `idx` from the pool to the training set."""
    data['train']['X'] = np.append(data['train']['X'], np.atleast_2d(data['pool']['X'][idx]), axis=0)
    data['train']['y'] = np.append(data['train']['y'], np.atleast_1d(data['pool']['y'][idx]), axis=0)    
    data['pool']['X'] = np.delete(data['pool']['X'], idx, axis=0)
    data['pool']['y'] = np.delete(data['pool']['y'], idx, axis=0)


def update_data(data, idx):
   
    data['train']['X'] = np.append(data['train']['X'], np.atleast_2d(data['pool']['X'][idx]), axis=0)
    data['train']['y'] = np.append(data['train']['y'], np.atleast_1d(data['pool']['y'][idx]), axis=0)    
    data['pool']['X'] = np.delete(data['pool']['X'], idx, axis=0)
    data['pool']['y'] = np.delete(data['pool']['y'], idx, axis=0)


### Updates model based on strategy and paradigm

In [53]:
from tqdm import tqdm
def fit_model(paradigm, strategy, n_init, n_iterations, use_classes=None, use_features=None, plot=False):
    
    scores = np.zeros(n_iterations)
    model = LogisticRegression(C=1e1, solver='sag')



    for i in range(n_iterations):

        model = model.fit(data['train']['X'], data['train']['y'])


        prob = model.predict_proba(data['pool']['X'])
        scores[i] = model.score(data['test']['X'], data['test']['y'])

    
        if i < n_iterations:
            if paradigm == 'active learning':
                uncertainty = evaluate_uncertainty(prob, strategy)
                if strategy in ('least confident', 'entropy'):
                    idx = uncertainty.argmax()
                elif strategy == 'maximum margin':
                    idx = uncertainty.argmin()
                else:
                    raise ValueError
            elif paradigm == 'random':
                uncertainty = None
                idx = np.random.choice(np.arange(len(data['pool']['X'])))
            else:
                raise ValueError
            if plot:
                plot_pool(data, idx, uncertainty)
                print("Get plottet")
          
            update_data(data, idx)
    return scores

### Evaluate model

In [ ]:
n_init = 5
n_iterations = 50 - n_init
n_avg = 50

# Average `n_avg` fits
scores_al = np.zeros((n_avg, n_iterations))
scores_rn = np.zeros((n_avg, n_iterations))

# solution::start
for i in tqdm(range(n_avg)):
    scores_al[i] = fit_model('active learning', 'entropy', n_init, n_iterations)
    scores_rn[i] = fit_model('random', 'entropy', n_init, n_iterations)

# Plot the results
fig, ax = plt.subplots(1, 1)
ax.plot(np.arange(n_init, n_iterations+n_init), scores_al.mean(0))
ax.plot(np.arange(n_init, n_iterations+n_init), scores_rn.mean(0))
ax.legend(['active learning', 'random'])
ax.set_xlabel('Training set size')
ax.set_ylabel('Classification Accuracy')

  0%|          | 0/50 [00:00<?, ?it/s]/Users/kento/Documents/DTU/Semester04/02463 Active machine learning and agency/Active-Machine-Learning-Imbalanced-Uncertainty-Sampling/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/kento/Documents/DTU/Semester04/02463 Active machine learning and agency/Active-Machine-Learning-Imbalanced-Uncertainty-Sampling/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/kento/Documents/DTU/Semester04/02463 Active machine learning and agency/Active-Machine-Learning-Imbalanced-Uncertainty-Sampling/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/kento/Documents/DTU/Semester04/02463 Act